[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn.functional as F

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    """
    x:      (B, C_in, H, W)
    weight: (C_out, C_in, kH, kW)
    """

    # Allow both integers and (height, width) tuples.
    if isinstance(stride, int):
        stride_h = stride_w = stride
    else:
        stride_h, stride_w = stride

    if isinstance(padding, int):
        pad_h = pad_w = padding
    else:
        pad_h, pad_w = padding

    B, C_in, H, W = x.shape
    C_out, C_weight, kH, kW = weight.shape

    assert C_in == C_weight
    assert stride_h > 0 and stride_w > 0
    assert pad_h >= 0 and pad_w >= 0

    # 1. Zero padding
    x = F.pad(
        x,
        (pad_w, pad_w, pad_h, pad_h)
    )

    # 2. Calculate output dimensions
    H_padded, W_padded = x.shape[2:]

    H_out = (H_padded - kH) // stride_h + 1
    W_out = (W_padded - kW) // stride_w + 1

    assert H_out > 0 and W_out > 0

    # 3. Initialize output
    out = x.new_zeros((B, C_out, H_out, W_out))

    # 4. Slide the kernel over spatial positions
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * stride_h
            w_start = j * stride_w

            # Extract a patch: (B, C_in, kH, kW)
            patch = x[
                :,
                :,
                h_start:h_start + kH,
                w_start:w_start + kW
            ]

            # Element-wise multiply and sum over
            # input channels and kernel dimensions.
            out[:, :, i, j] = (
                patch.unsqueeze(1) * weight.unsqueeze(0)
            ).sum(dim=(2, 3, 4))

    # 5. Add bias to each output channel
    if bias is not None:
        out = out + bias.view(1, C_out, 1, 1)

    return out



# def my_conv2d(x, weight, bias=None, stride=1, padding=0):
#     if padding > 0:
#         x = F.pad(x, [padding] * 4)
#     B, C_in, H, W = x.shape
#     C_out, _, kH, kW = weight.shape
#     H_out = (H - kH) // stride + 1
#     W_out = (W - kW) // stride + 1
#     patches = x.unfold(2, kH, stride).unfold(3, kW, stride)
#     out = torch.einsum('bihwjk,oijk->bohw', patches, weight)
#     if bias is not None:
#         out = out + bias.view(1, -1, 1, 1)
#     return out

In [4]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (5.6ms)
  ✅ [2/5] Matches F.conv2d (169.8ms)
  ✅ [3/5] With padding (17.1ms)
  ✅ [4/5] With stride (4.5ms)
  ✅ [5/5] Gradient flow (28.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (225.7ms total)
  Progress saved. Run status() to see your dashboard.



In [6]:
for stride, padding in [
    (1, 0),
    (1, 1),
    (2, 0),
    (2, 1),
]:
    actual = my_conv2d(
        x, w, stride=stride, padding=padding
    )
    expected = F.conv2d(
        x, w, stride=stride, padding=padding
    )

    print(
        stride,
        padding,
        actual.shape,
        torch.allclose(actual, expected, atol=1e-4)
    )

1 0 torch.Size([1, 16, 6, 6]) True
1 1 torch.Size([1, 16, 8, 8]) True
2 0 torch.Size([1, 16, 3, 3]) True
2 1 torch.Size([1, 16, 4, 4]) True
